In [1]:
%cd /content
!wget -c https://alphacephei.com/vosk-colab/kaldi.tar.gz
!tar xzf kaldi.tar.gz

/content
--2025-11-24 05:56:49--  https://alphacephei.com/vosk-colab/kaldi.tar.gz
Resolving alphacephei.com (alphacephei.com)... 188.40.21.16, 2a01:4f8:13a:279f::2
Connecting to alphacephei.com (alphacephei.com)|188.40.21.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1041599901 (993M) [application/octet-stream]
Saving to: ‘kaldi.tar.gz’

kaldi.tar.gz        100%[===================>] 993.35M  20.0MB/s    in 52s     

2025-11-24 05:57:41 (19.2 MB/s) - ‘kaldi.tar.gz’ saved [1041599901/1041599901]



In [2]:
%cd /content/kaldi/egs/ac
!wget -c https://alphacephei.com/vosk-colab/vosk-model-small-en-us-0.15-compile-colab.tar.gz
!rm -rf vosk-model-small-en-us-0.15-compile-colab
!tar xf vosk-model-small-en-us-0.15-compile-colab.tar.gz

/content/kaldi/egs/ac
--2025-11-24 05:58:54--  https://alphacephei.com/vosk-colab/vosk-model-small-en-us-0.15-compile-colab.tar.gz
Resolving alphacephei.com (alphacephei.com)... 188.40.21.16, 2a01:4f8:13a:279f::2
Connecting to alphacephei.com (alphacephei.com)|188.40.21.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59618100 (57M) [application/octet-stream]
Saving to: ‘vosk-model-small-en-us-0.15-compile-colab.tar.gz’

vosk-model-small-en 100%[===================>]  56.86M  17.9MB/s    in 3.7s    

2025-11-24 05:58:58 (15.2 MB/s) - ‘vosk-model-small-en-us-0.15-compile-colab.tar.gz’ saved [59618100/59618100]



In [9]:
%cd /content/kaldi/egs/ac/vosk-model-small-en-us-0.15-compile-colab
!ls
!cat compile-graph.sh
!bash compile-graph.sh

/content/kaldi/egs/ac/vosk-model-small-en-us-0.15-compile-colab
compile-graph.sh  db	     get_vocab.py  RESULTS
conf		  decode.sh  local	   steps
data		  dict.py    mfcc	   utils
data_test	  exp	     path.sh	   vosk_graph_adapt_20251124_063712.zip
#!/bin/bash

set -x

. path.sh

pip3 install phonetisaurus

rm -rf data
rm -rf exp/tdnn/lgraph
rm -rf exp/tdnn/lgraph_orig

mkdir -p data/dict
cp db/phone/* data/dict
./dict.py > data/dict/lexicon.txt

python3 ./get_vocab.py > data/mix.vocab
ngramsymbols data/mix.vocab data/mix.syms
farcompilestrings --fst_type=compact --symbols=data/mix.syms --keep_symbols --unknown_symbol="[unk]" db/extra.txt data/extra.far
ngramcount --order=3 data/extra.far - |
    ngramprint --integers | grep -v "<unk>" | ngramread |
    ngramshrink --method=count_prune --count_pattern="3+:3" |
    ngrammake --method=witten_bell - data/extra.mod
gunzip -c db/en-50k-0.4-android.lm.gz | ngramread --renormalize_arpa --ARPA --symbols=data/mix.syms - data/en-us.mod
ngrammerge

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
!cat decode.sh
!bash decode.sh

#!/bin/bash

. path.sh

steps/make_mfcc.sh --nj 10 data_test/test_small exp/make_mfcc/test mfcc
steps/compute_cmvn_stats.sh data_test/test_small exp/make_mfcc/test mfcc
utils/fix_data_dir.sh data_test/test_small

steps/online/nnet2/extract_ivectors_online.sh --nj 4 \
          data_test/test_small exp/extractor \
          exp/ivectors_test

steps/nnet3/decode.sh --nj 4 \
          --acwt 1.0 --post-decode-acwt 10.0 \
          --online-ivector-dir exp/ivectors_test \
          exp/tdnn/graph_adapt data_test/test_small exp/tdnn/decode_test_adapt

steps/nnet3/decode.sh --nj 4 \
          --acwt 1.0 --post-decode-acwt 10.0 \
          --online-ivector-dir exp/ivectors_test \
          exp/tdnn/graph data_test/test_small exp/tdnn/decode_test

#steps/nnet3/decode_lookahead.sh --nj 4 \
#          --acwt 1.0 --post-decode-acwt 10.0 \
#          --online-ivector-dir exp/ivectors_test \
#          exp/tdnn/lgraph data_test/test_small exp/tdnn/decode_test_adapt
#steps/nnet3/decode_lookahead.sh 

In [11]:
# Export the adapted graph to ZIP file (Colab compatible)
import os
import zipfile
from datetime import datetime

# For Google Colab environment
print("🔍 Detecting environment...")
try:
    from google.colab import files
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✅ Running locally")

# Colab uses /content, local uses our work directory
if IN_COLAB:
    model_dir = "/content/kaldi/egs/ac/vosk-model-small-en-us-0.15-compile-colab"
else:
    model_dir = "/Users/fibo-mac-501/Documents/medbrain/data/vosk/adaptation/work/kaldi/egs/ac/vosk-model-small-en-us-0.15-compile-colab"

os.chdir(model_dir)
print(f"Working in: {os.getcwd()}")

# Check if graph_adapt exists
graph_adapt_path = "exp/tdnn/graph_adapt"
if os.path.exists(graph_adapt_path):
    print(f"✅ Found adapted graph at: {graph_adapt_path}")

    # Create timestamp for unique filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_zip = f"vosk_graph_adapt_{timestamp}.zip"

    print(f"📦 Creating ZIP file: {output_zip}")

    # Create ZIP file with the adapted graph
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through the graph_adapt directory
        for root, dirs, files in os.walk(graph_adapt_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Calculate relative path from the graph_adapt directory
                relative_path = os.path.relpath(file_path, graph_adapt_path)
                zipf.write(file_path, f"graph_adapt/{relative_path}")
                print(f"  ✓ Added: {relative_path}")

    # Get file size
    zip_size = os.path.getsize(output_zip)
    print(f"\n✅ Successfully created: {output_zip}")
    print(f"📊 File size: {zip_size / (1024*1024):.2f} MB")
    print(f"📁 Full path: {os.path.abspath(output_zip)}")

    # List contents of the ZIP for verification
    print(f"\n📋 ZIP contents (first 10 files):")
    with zipfile.ZipFile(output_zip, 'r') as zipf:
        files_list = zipf.namelist()
        for name in files_list[:10]:
            print(f"  - {name}")
        if len(files_list) > 10:
            print(f"  ... and {len(files_list) - 10} more files")
        print(f"📊 Total files in ZIP: {len(files_list)}")

    # Download file in Colab or show local path
    if IN_COLAB:
        print(f"\n⬇️ Downloading {output_zip} to your computer...")
        try:
            files.download(output_zip)
            print("✅ Download started! Check your browser downloads.")
        except Exception as e:
            print(f"❌ Download failed: {e}")
            print(f"💡 You can manually download from: {os.path.abspath(output_zip)}")
    else:
        print(f"\n💾 File saved locally at: {os.path.abspath(output_zip)}")
        print("💡 You can copy this file to your Vosk model directory")

    # Instructions for using the adapted graph
    print(f"\n📖 Usage Instructions:")
    print(f"1. Extract the ZIP file to get the 'graph_adapt' directory")
    print(f"2. Copy 'graph_adapt' to your Vosk model directory")
    print(f"3. Use it in your Vosk recognizer as the graph parameter")
    print(f"4. This adapted graph includes your custom vocabulary/language model")

else:
    print(f"❌ Adapted graph not found at: {graph_adapt_path}")
    print("❗ Make sure the compilation completed successfully first.")

    # List what's actually in exp/tdnn/
    exp_tdnn_path = "exp/tdnn"
    if os.path.exists(exp_tdnn_path):
        print(f"\n📁 Contents of {exp_tdnn_path}:")
        for item in os.listdir(exp_tdnn_path):
            item_path = os.path.join(exp_tdnn_path, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}/")
            else:
                print(f"  📄 {item}")
    else:
        print(f"❌ {exp_tdnn_path} does not exist")
        print("💡 Run the compilation cell first to generate the adapted graph")

🔍 Detecting environment...
✅ Running in Google Colab
Working in: /content/kaldi/egs/ac/vosk-model-small-en-us-0.15-compile-colab
✅ Found adapted graph at: exp/tdnn/graph_adapt
📦 Creating ZIP file: vosk_graph_adapt_20251124_065223.zip
  ✓ Added: num_pdfs
  ✓ Added: HCLG.fst
  ✓ Added: words.txt
  ✓ Added: disambig_tid.int
  ✓ Added: phones.txt
  ✓ Added: phones/silence.csl
  ✓ Added: phones/optional_silence.csl
  ✓ Added: phones/optional_silence.int
  ✓ Added: phones/align_lexicon.txt
  ✓ Added: phones/word_boundary.txt
  ✓ Added: phones/align_lexicon.int
  ✓ Added: phones/disambig.int
  ✓ Added: phones/disambig.txt
  ✓ Added: phones/optional_silence.txt
  ✓ Added: phones/word_boundary.int

✅ Successfully created: vosk_graph_adapt_20251124_065223.zip
📊 File size: 63.76 MB
📁 Full path: /content/kaldi/egs/ac/vosk-model-small-en-us-0.15-compile-colab/vosk_graph_adapt_20251124_065223.zip

📋 ZIP contents (first 10 files):
  - graph_adapt/num_pdfs
  - graph_adapt/HCLG.fst
  - graph_adapt/word

In [7]:
!bash RESULTS